## Quantum Fourier Transform & Algebraic Algorithms
These leverage the [Quantum Fourier Transform (QFT)](https://en.wikipedia.org/wiki/Quantum_Fourier_transform) to convert state computational bases into [phase space](https://en.wikipedia.org/wiki/Phase_space), solving period-finding and factoring tasks with exponential speedup.

### 1. Quantum Fourier Transform (QFT)
* **Problem Domain:** [Frequency Analysis](https://en.wikipedia.org/wiki/Frequency_analysis), Phase Manipulation.
* **Core Function:** Maps a computational state $\vert{}x\rangle$ to a Fourier basis state $\frac{1}{\sqrt{N}}\sum_{y=0}^{N-1} e^{2\pi i x y / N}\vert{}y\rangle$.
* **Algorithmic Mechanism:**
    1. Apply Hadamard gates sequentially to each qubit.
    2. Apply controlled phase rotations $CP(\theta_k)$ with decreasing angles $\theta_k = \frac{2\pi}{2^k}$.
    3. Swap qubit order at the end to align bit significance.

In [1]:
import numpy as np
from qiskit import QuantumCircuit

def create_qft(num_qubits):
    qc = QuantumCircuit(num_qubits, name="QFT")
    for i in range(num_qubits):
        qc.h(i)
        for j in range(i + 1, num_qubits):
            angle = np.pi / (2 ** (j - i))
            qc.cp(angle, j, i)
    # Swap qubits for correct bit ordering
    for i in range(num_qubits // 2):
        qc.swap(i, num_qubits - 1 - i)
    return qc

qft_3q = create_qft(3)
print("3-Qubit QFT Circuit:")
print(qft_3q.draw('text'))

3-Qubit QFT Circuit:
     ┌───┐                                        
q_0: ┤ H ├─■────────■───────────────────────────X─
     └───┘ │P(π/2)  │       ┌───┐               │ 
q_1: ──────■────────┼───────┤ H ├─■─────────────┼─
                    │P(π/4) └───┘ │P(π/2) ┌───┐ │ 
q_2: ───────────────■─────────────■───────┤ H ├─X─
                                          └───┘   


### 2. [Quantum Phase Estimation (QPE)](https://en.wikipedia.org/wiki/Quantum_phase_estimation_algorithm)
* **Problem Domain:** [Eigenvalue Determination](https://en.wikipedia.org/wiki/Eigenvalues_and_eigenvectors), [Quantum Chemistry](https://en.wikipedia.org/wiki/Quantum_chemistry).
* **Core Function:** Estimates phase $\theta$ of a unitary operator $U$ given eigenvector $\vert{}\psi\rangle$ such that $U\vert{}\psi\rangle = e^{2\pi i \theta}\vert{}\psi\rangle$.
* **Algorithmic Mechanism:**
    1. Initialize counting register in superposition and target register in state $\vert{}\psi\rangle$.
    2. Apply controlled unitary powers $C-U^{2^j}$ from counting qubits to the target register.
    3. Apply Inverse QFT ($QFT^\dagger$) to the counting register.
    4. Measure counting qubits to read phase $\theta$.

In [4]:
import numpy as np
from qiskit import QuantumCircuit

# QPE for T-gate (unitary phase theta = 1/8)
qc = QuantumCircuit(4, 3) # 3 counting qubits, 1 target qubit

# Initialize target in state |1>
qc.x(3)

# Step 1: Superposition on counting qubits
qc.h([0, 1, 2])

# Step 2: Controlled-U operations (T-gate powers)
qc.cp(np.pi / 4, 0, 3)          # C-U^(2^0)
qc.cp(2 * np.pi / 4, 1, 3)      # C-U^(2^1)
qc.cp(4 * np.pi / 4, 2, 3)      # C-U^(2^2)

# Step 3: Inverse QFT on counting qubits (simplified)
qc.h(0)
qc.cp(-np.pi / 2, 0, 1)
qc.h(1)
qc.cp(-np.pi / 4, 0, 2)
qc.cp(-np.pi / 2, 1, 2)
qc.h(2)

qc.measure([0, 1, 2], [0, 1, 2])

print("Quantum Phase Estimation Circuit:")
print(qc.draw('text'))

Quantum Phase Estimation Circuit:
     ┌───┐           ┌───┐                                     ┌─┐           
q_0: ┤ H ├─■─────────┤ H ├───■──────────────■──────────────────┤M├───────────
     ├───┤ │         └───┘   │P(-π/2) ┌───┐ │                  └╥┘     ┌─┐   
q_1: ┤ H ├─┼────────■────────■────────┤ H ├─┼─────────■─────────╫──────┤M├───
     ├───┤ │        │                 └───┘ │P(-π/4)  │P(-π/2)  ║ ┌───┐└╥┘┌─┐
q_2: ┤ H ├─┼────────┼─────────■─────────────■─────────■─────────╫─┤ H ├─╫─┤M├
     ├───┤ │P(π/4)  │P(π/2)   │P(π)                             ║ └───┘ ║ └╥┘
q_3: ┤ X ├─■────────■─────────■─────────────────────────────────╫───────╫──╫─
     └───┘                                                      ║       ║  ║ 
c: 3/═══════════════════════════════════════════════════════════╩═══════╩══╩═
                                                                0       1  2 
